# Optimisation de la Coloration de Cartes Géographiques
## Recherche Tabou hybridée avec Machine Learning pour le choix des opérateurs de voisinage

**M1 MIAGE MIXTE — Projet Optimisation Combinatoire**

---

### Structure du notebook
1. Imports & Configuration
2. Modélisation du problème
3. Recherche Tabou de base (sans ML)
4. Collecte de données & feature engineering
5. Entraînement du classifieur ML
6. Recherche Tabou hybride (avec ML)
7. Expérimentations & comparaisons
8. Analyse des résultats & discussion

---
## 1. Imports & Configuration

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import time
import copy
from collections import deque, defaultdict
from itertools import combinations

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("✅ Imports OK")
print(f"   NetworkX  : {nx.__version__}")
print(f"   NumPy     : {np.__version__}")
print(f"   Pandas    : {pd.__version__}")

---
## 2. Modélisation du problème

### 2.1 Le problème de coloration de graphe

Le **problème de coloration de graphe** consiste à assigner une couleur à chaque nœud d'un graphe G=(V,E) tel que deux nœuds adjacents n'aient jamais la même couleur, en **minimisant le nombre de couleurs utilisées** (nombre chromatique χ(G)).

**Application cartographique** : chaque région est un nœud, deux régions voisines sont reliées par une arête. Le théorème des 4 couleurs garantit que toute carte planaire est colorable avec au plus 4 couleurs.

**Fonction objectif** :
$$f(s) = \alpha \cdot \text{conflits}(s) + \beta \cdot \text{nb\_couleurs}(s)$$

où `conflits(s)` = nombre de paires de nœuds adjacents ayant la même couleur.

In [ ]:
# ============================================================
# 2.2 Génération des instances de graphe
# ============================================================

def generate_map_graph(n_regions=20, seed=SEED):
    """
    Génère un graphe planaire simulant une carte géographique.
    Utilise un graphe de Delaunay approximé via positions aléatoires.
    """
    np.random.seed(seed)
    # Positions aléatoires des capitales de régions
    pos = {i: (np.random.uniform(0, 10), np.random.uniform(0, 10))
           for i in range(n_regions)}

    G = nx.random_geometric_graph(n_regions, radius=3.5, seed=seed)
    # Assurer la connexité
    if not nx.is_connected(G):
        components = list(nx.connected_components(G))
        for i in range(len(components) - 1):
            u = list(components[i])[0]
            v = list(components[i+1])[0]
            G.add_edge(u, v)
    return G


def generate_dsjc_like(n, p, seed=SEED):
    """
    Génère un graphe aléatoire de type DSJC (benchmark classique).
    n = nombre de nœuds, p = probabilité d'arête.
    """
    G = nx.erdos_renyi_graph(n, p, seed=seed)
    if not nx.is_connected(G):
        components = list(nx.connected_components(G))
        for i in range(len(components) - 1):
            u = list(components[i])[0]
            v = list(components[i+1])[0]
            G.add_edge(u, v)
    return G


# Instances de test
G_small  = generate_map_graph(n_regions=15, seed=42)   # petit
G_medium = generate_map_graph(n_regions=30, seed=43)   # moyen
G_large  = generate_map_graph(n_regions=50, seed=44)   # grand
G_dsjc   = generate_dsjc_like(n=50, p=0.3, seed=42)   # benchmark DSJC

print("Instances générées :")
for name, G in [("G_small", G_small), ("G_medium", G_medium),
                ("G_large", G_large), ("G_dsjc", G_dsjc)]:
    density = nx.density(G)
    print(f"  {name:10s} | nœuds={G.number_of_nodes():3d} "
          f"| arêtes={G.number_of_edges():4d} "
          f"| densité={density:.3f}")

In [ ]:
# ============================================================
# 2.3 Visualisation d'une carte
# ============================================================

def visualize_coloring(G, coloring, title="Coloration du graphe"):
    """
    Affiche le graphe avec sa coloration.
    coloring : dict {nœud: couleur (int)}
    """
    palette = plt.cm.get_cmap('tab10', max(coloring.values()) + 1)
    node_colors = [palette(coloring[n]) for n in G.nodes()]

    # Arêtes en conflit en rouge
    conflict_edges = [(u, v) for u, v in G.edges()
                      if coloring[u] == coloring[v]]
    normal_edges   = [(u, v) for u, v in G.edges()
                      if coloring[u] != coloring[v]]

    pos = nx.spring_layout(G, seed=SEED)
    fig, ax = plt.subplots(figsize=(9, 6))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                           node_size=400, ax=ax)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
    nx.draw_networkx_edges(G, pos, edgelist=normal_edges,
                           edge_color='gray', ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=conflict_edges,
                           edge_color='red', width=2.5, ax=ax)

    n_conflicts = len(conflict_edges)
    n_colors    = len(set(coloring.values()))
    ax.set_title(f"{title}\nCouleurs={n_colors} | Conflits={n_conflicts}",
                 fontsize=12)
    if conflict_edges:
        ax.legend(handles=[
            mpatches.Patch(color='red', label=f'{n_conflicts} conflit(s)')
        ])
    plt.tight_layout()
    plt.show()


# Solution initiale greedy pour visualisation
greedy_coloring = nx.coloring.greedy_color(G_small, strategy='largest_first')
visualize_coloring(G_small, greedy_coloring, "Coloration greedy initiale (G_small)")

In [ ]:
# ============================================================
# 2.4 Fonctions utilitaires : solution, conflits, objectif
# ============================================================

def count_conflicts(G, coloring):
    """Nombre d'arêtes dont les deux extrémités ont la même couleur."""
    return sum(1 for u, v in G.edges() if coloring[u] == coloring[v])


def objective(G, coloring, alpha=10, beta=1):
    """
    Fonction objectif à minimiser.
    alpha * conflits + beta * nb_couleurs
    On pénalise fortement les conflits, et on minimise le nb de couleurs.
    """
    conflicts  = count_conflicts(G, coloring)
    n_colors   = len(set(coloring.values()))
    return alpha * conflicts + beta * n_colors


def initial_solution(G, n_colors=None):
    """
    Solution initiale par algorithme greedy (largest_first).
    Si n_colors est fourni, on force l'usage de k couleurs (peut créer des conflits).
    """
    greedy = nx.coloring.greedy_color(G, strategy='largest_first')
    if n_colors is None:
        return dict(greedy)
    # Forcer k couleurs → certains nœuds peuvent être en conflit
    return {node: color % n_colors for node, color in greedy.items()}


def solution_info(G, coloring):
    """Affiche un résumé de la solution."""
    c = count_conflicts(G, coloring)
    k = len(set(coloring.values()))
    print(f"  Conflits    : {c}")
    print(f"  Nb couleurs : {k}")
    print(f"  Objectif    : {objective(G, coloring)}")


sol_init = initial_solution(G_small, n_colors=4)
print("Solution initiale (forcée à 4 couleurs) :")
solution_info(G_small, sol_init)

---
## 3. Les trois opérateurs de voisinage

### 3.1 Recolor
Changer la couleur d'un seul nœud en conflit.

### 3.2 Swap  
Échanger les couleurs de deux nœuds.

### 3.3 Kempe Chain ⭐
Inverser deux couleurs i↔j dans une composante connexe restreinte aux nœuds de couleur i ou j.

In [ ]:
# ============================================================
# 3.1 Opérateur RECOLOR
# ============================================================

def op_recolor(G, coloring, n_colors, tabu_list):
    """
    Génère le meilleur voisin par recoloration d'un nœud en conflit.
    Retourne (nouvelle_coloring, mouvement, delta_objectif)
    mouvement = (nœud, ancienne_couleur, nouvelle_couleur)
    """
    best_neighbor = None
    best_delta    = float('inf')
    best_move     = None

    # Nœuds en conflit
    conflict_nodes = [u for u in G.nodes()
                      if any(coloring[u] == coloring[v]
                             for v in G.neighbors(u))]
    if not conflict_nodes:
        conflict_nodes = list(G.nodes())  # si pas de conflit → tous les nœuds

    current_obj = objective(G, coloring)

    for node in conflict_nodes:
        old_color = coloring[node]
        for new_color in range(n_colors):
            if new_color == old_color:
                continue
            move = (node, old_color, new_color)
            # Vérifier tabou
            if move in tabu_list:
                continue
            # Évaluer
            new_col = dict(coloring)
            new_col[node] = new_color
            delta = objective(G, new_col) - current_obj
            if delta < best_delta:
                best_delta    = delta
                best_neighbor = new_col
                best_move     = move

    return best_neighbor, best_move, best_delta


# ============================================================
# 3.2 Opérateur SWAP
# ============================================================

def op_swap(G, coloring, n_colors, tabu_list, n_candidates=20):
    """
    Génère le meilleur voisin par échange de couleurs entre deux nœuds.
    n_candidates = nb de paires testées (pour limiter le temps de calcul).
    """
    best_neighbor = None
    best_delta    = float('inf')
    best_move     = None

    current_obj = objective(G, coloring)
    nodes = list(G.nodes())

    # Échantillonner des paires aléatoires
    pairs = [random.sample(nodes, 2) for _ in range(n_candidates)]

    for u, v in pairs:
        if coloring[u] == coloring[v]:
            continue
        move = ('swap', u, v, coloring[u], coloring[v])
        if move in tabu_list:
            continue
        new_col = dict(coloring)
        new_col[u], new_col[v] = coloring[v], coloring[u]
        delta = objective(G, new_col) - current_obj
        if delta < best_delta:
            best_delta    = delta
            best_neighbor = new_col
            best_move     = move

    return best_neighbor, best_move, best_delta


print("✅ Opérateurs Recolor et Swap définis")

In [ ]:
# ============================================================
# 3.3 Opérateur KEMPE CHAIN ⭐
# ============================================================

def get_kempe_chain(G, coloring, start_node, color_i, color_j):
    """
    Trouve la chaîne de Kempe contenant start_node pour les couleurs i et j.
    
    Une chaîne de Kempe(i,j) à partir d'un nœud n est la composante connexe
    dans le sous-graphe induit par les nœuds de couleur i ou j,
    contenant n.
    
    Algorithme : BFS dans le sous-graphe {nœuds de couleur i ou j}
    """
    if coloring[start_node] not in (color_i, color_j):
        return set()

    chain   = set()
    queue   = deque([start_node])
    visited = {start_node}

    while queue:
        node = queue.popleft()
        chain.add(node)
        for neighbor in G.neighbors(node):
            if neighbor not in visited and coloring[neighbor] in (color_i, color_j):
                visited.add(neighbor)
                queue.append(neighbor)

    return chain


def apply_kempe_chain(coloring, chain, color_i, color_j):
    """
    Applique l'inversion de Kempe : inverse color_i <-> color_j
    pour tous les nœuds de la chaîne.
    """
    new_col = dict(coloring)
    for node in chain:
        if new_col[node] == color_i:
            new_col[node] = color_j
        elif new_col[node] == color_j:
            new_col[node] = color_i
    return new_col


def op_kempe_chain(G, coloring, n_colors, tabu_list, n_candidates=15):
    """
    Génère le meilleur voisin par inversion de chaîne de Kempe.
    Teste n_candidates combinaisons (nœud_départ, couleur_i, couleur_j).
    """
    best_neighbor  = None
    best_delta     = float('inf')
    best_move      = None
    best_chain_len = 0

    current_obj = objective(G, coloring)
    nodes       = list(G.nodes())
    color_pairs = list(combinations(range(n_colors), 2))

    # Priorité aux nœuds en conflit
    conflict_nodes = [u for u in G.nodes()
                      if any(coloring[u] == coloring[v]
                             for v in G.neighbors(u))]
    candidate_nodes = conflict_nodes if conflict_nodes else nodes

    attempts = 0
    random.shuffle(candidate_nodes)
    random.shuffle(color_pairs)

    for start in candidate_nodes:
        for (ci, cj) in color_pairs:
            if attempts >= n_candidates:
                break
            if coloring[start] not in (ci, cj):
                continue

            chain = get_kempe_chain(G, coloring, start, ci, cj)
            if len(chain) < 2:   # chaîne triviale
                continue

            move = ('kempe', start, ci, cj, len(chain))
            if move[:4] in [(m[:4] if isinstance(m, tuple) and m[0]=='kempe'
                             else m) for m in tabu_list]:
                continue

            new_col = apply_kempe_chain(coloring, chain, ci, cj)
            delta   = objective(G, new_col) - current_obj

            if delta < best_delta:
                best_delta     = delta
                best_neighbor  = new_col
                best_move      = move
                best_chain_len = len(chain)

            attempts += 1
        if attempts >= n_candidates:
            break

    return best_neighbor, best_move, best_delta, best_chain_len


# --- Test rapide de Kempe Chain ---
test_sol = initial_solution(G_small, n_colors=4)
print("Test Kempe Chain sur G_small :")
print(f"  Conflits avant : {count_conflicts(G_small, test_sol)}")
result, move, delta, clen = op_kempe_chain(G_small, test_sol, 4, set())
if result:
    print(f"  Meilleur delta : {delta:.2f}")
    print(f"  Taille chaîne  : {clen}")
    print(f"  Conflits après : {count_conflicts(G_small, result)}")
else:
    print("  Aucun mouvement Kempe trouvé")

---
## 4. Recherche Tabou de base (sans ML)

La sélection de l'opérateur est ici **aléatoire uniforme** (baseline).

In [ ]:
# ============================================================
# 4.1 Recherche Tabou classique
# ============================================================

def tabu_search(
    G,
    n_colors,
    max_iter      = 500,
    tabu_tenure   = 10,
    operator_mode = 'random',   # 'random' | 'recolor' | 'swap' | 'kempe' | 'ml'
    ml_model      = None,       # classifieur ML (si operator_mode='ml')
    scaler        = None,
    verbose       = False
):
    """
    Recherche Tabou pour la coloration de graphe.
    
    Paramètres :
    - G            : graphe NetworkX
    - n_colors     : nombre de couleurs autorisées
    - max_iter     : nombre max d'itérations
    - tabu_tenure  : durée de vie d'un mouvement dans la liste tabou
    - operator_mode: stratégie de sélection d'opérateur
    - ml_model     : modèle ML pour prédire l'opérateur (mode 'ml')
    
    Retourne un dictionnaire de résultats.
    """
    # --- Initialisation ---
    current     = initial_solution(G, n_colors)
    best        = dict(current)
    best_obj    = objective(G, best)
    tabu_list   = deque(maxlen=tabu_tenure * G.number_of_nodes())

    # Historique pour analyse
    history = {
        'obj'        : [],
        'conflicts'  : [],
        'n_colors'   : [],
        'operators'  : [],
        'features'   : []       # features ML collectées
    }

    no_improve_count = 0
    operators = ['recolor', 'swap', 'kempe']

    for it in range(max_iter):
        current_obj      = objective(G, current)
        current_conflicts= count_conflicts(G, current)

        # --- Features de l'état courant (pour ML) ---
        conflict_ratio = current_conflicts / max(G.number_of_edges(), 1)
        color_counts   = defaultdict(int)
        for c in current.values():
            color_counts[c] += 1
        color_entropy  = -sum((v/G.number_of_nodes()) *
                               np.log(v/G.number_of_nodes() + 1e-9)
                               for v in color_counts.values())
        features = [
            conflict_ratio,                          # ratio conflits/arêtes
            no_improve_count / max(max_iter, 1),    # plateau normalisé
            nx.density(G),                           # densité du graphe
            color_entropy,                           # entropie des couleurs
            len(set(current.values())) / n_colors,  # taux d'utilisation couleurs
            current_conflicts / max(G.number_of_nodes(), 1),  # conflits/nœud
        ]
        history['features'].append(features)

        # --- Sélection de l'opérateur ---
        if operator_mode == 'random':
            op = random.choice(operators)
        elif operator_mode in operators:
            op = operator_mode
        elif operator_mode == 'ml' and ml_model is not None:
            feat_scaled = scaler.transform([features])
            op_idx = ml_model.predict(feat_scaled)[0]
            op = operators[op_idx]
        else:
            # Heuristique simple : si plateau → kempe, sinon recolor
            op = 'kempe' if no_improve_count > 20 else 'recolor'

        # --- Application de l'opérateur ---
        chain_len = 0
        if op == 'recolor':
            neighbor, move, delta = op_recolor(G, current, n_colors, tabu_list)
        elif op == 'swap':
            neighbor, move, delta = op_swap(G, current, n_colors, tabu_list)
        else:  # kempe
            neighbor, move, delta, chain_len = op_kempe_chain(
                G, current, n_colors, tabu_list)

        history['operators'].append(op)

        # --- Mise à jour si mouvement valide ---
        if neighbor is None:
            # Aucun mouvement non-tabou → forcer recolor
            neighbor, move, delta = op_recolor(
                G, current, n_colors, set())
            if neighbor is None:
                break

        # Critère d'aspiration : accepter même si tabou si meilleur que best
        new_obj = objective(G, neighbor)
        if new_obj < best_obj or move not in tabu_list:
            current = neighbor
            if move:
                tabu_list.append(move)

        # Mise à jour du meilleur
        if new_obj < best_obj:
            best     = dict(current)
            best_obj = new_obj
            no_improve_count = 0
        else:
            no_improve_count += 1

        # Enregistrement historique
        history['obj'].append(new_obj)
        history['conflicts'].append(count_conflicts(G, current))
        history['n_colors'].append(len(set(current.values())))

        if verbose and it % 100 == 0:
            print(f"  iter={it:4d} | obj={new_obj:.1f} "
                  f"| conflicts={count_conflicts(G,current)} "
                  f"| op={op:7s} | plateau={no_improve_count}")

        # Arrêt si solution parfaite
        if count_conflicts(G, best) == 0:
            if verbose:
                print(f"  ✅ Solution sans conflit trouvée à l'itération {it}")
            break

    return {
        'best_coloring' : best,
        'best_obj'      : best_obj,
        'best_conflicts': count_conflicts(G, best),
        'best_n_colors' : len(set(best.values())),
        'history'       : history,
        'n_iter'        : it + 1
    }


print("✅ Fonction tabu_search définie")

In [ ]:
# ============================================================
# 4.2 Test de la Recherche Tabou de base
# ============================================================

print("=" * 60)
print("Recherche Tabou de base — G_medium, 4 couleurs")
print("=" * 60)

result_base = tabu_search(
    G_medium,
    n_colors    = 4,
    max_iter    = 500,
    tabu_tenure = 10,
    operator_mode = 'random',
    verbose     = True
)

print("\nRésultat final :")
print(f"  Conflits    : {result_base['best_conflicts']}")
print(f"  Nb couleurs : {result_base['best_n_colors']}")
print(f"  Objectif    : {result_base['best_obj']:.2f}")
print(f"  Itérations  : {result_base['n_iter']}")

visualize_coloring(G_medium, result_base['best_coloring'],
                   "Tabou de base — G_medium")

---
## 5. Collecte de données pour le Machine Learning

On lance plusieurs runs de Recherche Tabou avec un opérateur forcé à chaque itération, et on enregistre :
- les **features** de l'état courant
- l'**opérateur** qui a donné le meilleur résultat (label)

In [ ]:
# ============================================================
# 5.1 Génération du dataset d'entraînement
# ============================================================

def collect_training_data(graphs, n_colors=4, n_runs_per_graph=5,
                           max_iter=300):
    """
    Pour chaque graphe, on exécute la Recherche Tabou avec les 3 opérateurs
    et on collecte : features de l'état + quel opérateur a donné le
    meilleur delta à cette itération.

    Label : 0=recolor, 1=swap, 2=kempe
    """
    X, y = [], []
    operator_map = {'recolor': 0, 'swap': 1, 'kempe': 2}

    for G in graphs:
        for run in range(n_runs_per_graph):
            seed_run = SEED + run
            random.seed(seed_run)
            np.random.seed(seed_run)

            current   = initial_solution(G, n_colors)
            tabu_list = deque(maxlen=10 * G.number_of_nodes())
            no_improve= 0

            for it in range(max_iter):
                current_obj   = objective(G, current)
                n_conflicts   = count_conflicts(G, current)

                # Features
                conflict_ratio = n_conflicts / max(G.number_of_edges(), 1)
                color_counts   = defaultdict(int)
                for c in current.values():
                    color_counts[c] += 1
                color_entropy = -sum(
                    (v/G.number_of_nodes()) *
                    np.log(v/G.number_of_nodes() + 1e-9)
                    for v in color_counts.values()
                )
                features = [
                    conflict_ratio,
                    no_improve / max(max_iter, 1),
                    nx.density(G),
                    color_entropy,
                    len(set(current.values())) / n_colors,
                    n_conflicts / max(G.number_of_nodes(), 1),
                ]

                # Tester les 3 opérateurs et choisir le meilleur
                results = {}
                r, _, d     = op_recolor(G, current, n_colors, tabu_list)
                results['recolor'] = d if r is not None else float('inf')

                r, _, d     = op_swap(G, current, n_colors, tabu_list)
                results['swap']    = d if r is not None else float('inf')

                r, _, d, _  = op_kempe_chain(G, current, n_colors, tabu_list)
                results['kempe']   = d if r is not None else float('inf')

                best_op = min(results, key=results.get)
                if results[best_op] == float('inf'):
                    break

                X.append(features)
                y.append(operator_map[best_op])

                # Appliquer le meilleur opérateur pour continuer
                if best_op == 'recolor':
                    neighbor, move, _ = op_recolor(
                        G, current, n_colors, tabu_list)
                elif best_op == 'swap':
                    neighbor, move, _ = op_swap(
                        G, current, n_colors, tabu_list)
                else:
                    neighbor, move, _, _ = op_kempe_chain(
                        G, current, n_colors, tabu_list)

                if neighbor is None:
                    break

                new_obj = objective(G, neighbor)
                if new_obj < current_obj:
                    no_improve = 0
                else:
                    no_improve += 1

                current = neighbor
                if move:
                    tabu_list.append(move)

                if count_conflicts(G, current) == 0:
                    break

    return np.array(X), np.array(y)


# Graphes d'entraînement (variété de tailles)
train_graphs = [
    generate_map_graph(n_regions=n, seed=s)
    for n, s in [(15,1),(20,2),(25,3),(30,4),(35,5),(40,6)]
]

print("Collecte des données d'entraînement...")
t0 = time.time()
X_train_raw, y_train_raw = collect_training_data(
    train_graphs, n_colors=4, n_runs_per_graph=3, max_iter=200)
print(f"  Terminé en {time.time()-t0:.1f}s")
print(f"  Échantillons collectés : {len(X_train_raw)}")
print(f"  Distribution des labels : "
      f"recolor={sum(y_train_raw==0)} "
      f"swap={sum(y_train_raw==1)} "
      f"kempe={sum(y_train_raw==2)}")

In [ ]:
# ============================================================
# 5.2 Visualisation de la distribution des features
# ============================================================

feature_names = [
    'ratio_conflits', 'plateau_norm', 'densite_graphe',
    'entropie_couleurs', 'taux_utilisation_couleurs', 'conflits_par_noeud'
]

df_features = pd.DataFrame(X_train_raw, columns=feature_names)
df_features['operateur'] = ['recolor' if l==0 else 'swap' if l==1 else 'kempe'
                             for l in y_train_raw]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
colors_op = {'recolor': 'steelblue', 'swap': 'darkorange', 'kempe': 'green'}

for i, feat in enumerate(feature_names):
    for op, col in colors_op.items():
        subset = df_features[df_features['operateur'] == op][feat]
        axes[i].hist(subset, bins=30, alpha=0.5, color=col, label=op, density=True)
    axes[i].set_title(feat, fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle("Distribution des features par opérateur optimal", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Entraînement du classifieur ML (Random Forest)

In [ ]:
# ============================================================
# 6.1 Préparation & entraînement
# ============================================================

from sklearn.utils import resample

# Équilibrage des classes
df_bal = df_features.copy()
min_class = df_bal['operateur'].value_counts().min()
df_balanced = pd.concat([
    resample(df_bal[df_bal['operateur'] == op],
             replace=True, n_samples=min_class, random_state=SEED)
    for op in ['recolor', 'swap', 'kempe']
])

X_bal = df_balanced[feature_names].values
y_bal = df_balanced['operateur'].map({'recolor':0,'swap':1,'kempe':2}).values

# Split train/test
X_tr, X_te, y_tr, y_te = train_test_split(
    X_bal, y_bal, test_size=0.25, random_state=SEED, stratify=y_bal)

# Normalisation
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators = 200,
    max_depth    = 10,
    random_state = SEED,
    n_jobs       = -1
)
rf_model.fit(X_tr_s, y_tr)

# Évaluation
score_test = rf_model.score(X_te_s, y_te)
cv_scores  = cross_val_score(rf_model, X_tr_s, y_tr, cv=5)

print(f"Accuracy test  : {score_test:.3f}")
print(f"CV scores      : {cv_scores.round(3)} → mean={cv_scores.mean():.3f}")
print()
print("Rapport de classification :")
y_pred = rf_model.predict(X_te_s)
print(classification_report(y_te, y_pred,
      target_names=['recolor','swap','kempe']))

In [ ]:
# ============================================================
# 6.2 Matrice de confusion & importance des features
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Matrice de confusion
cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['recolor','swap','kempe'],
            yticklabels=['recolor','swap','kempe'])
ax1.set_title("Matrice de confusion", fontsize=12)
ax1.set_xlabel("Prédit")
ax1.set_ylabel("Réel")

# Importance des features
importances = rf_model.feature_importances_
ax2.barh(feature_names, importances, color='steelblue')
ax2.set_title("Importance des features (Random Forest)", fontsize=12)
ax2.set_xlabel("Importance")

plt.tight_layout()
plt.show()

---
## 7. Recherche Tabou hybride (avec ML)

On intègre le classifieur dans la boucle Tabou : à chaque itération, le modèle ML prédit l'opérateur optimal selon l'état courant.

In [ ]:
# ============================================================
# 7.1 Comparaison sur G_medium
# ============================================================

N_RUNS    = 10
MAX_ITER  = 500
N_COLORS  = 4

modes = ['random', 'recolor', 'swap', 'kempe', 'ml']
results_all = {m: [] for m in modes}

print("Comparaison des modes sur G_medium (10 runs chacun)...")

for mode in modes:
    for run in range(N_RUNS):
        random.seed(run)
        np.random.seed(run)
        t0 = time.time()
        res = tabu_search(
            G_medium,
            n_colors      = N_COLORS,
            max_iter      = MAX_ITER,
            tabu_tenure   = 10,
            operator_mode = mode,
            ml_model      = rf_model if mode == 'ml' else None,
            scaler        = scaler   if mode == 'ml' else None,
        )
        elapsed = time.time() - t0
        results_all[mode].append({
            'conflicts' : res['best_conflicts'],
            'n_colors'  : res['best_n_colors'],
            'obj'       : res['best_obj'],
            'time'      : elapsed,
            'n_iter'    : res['n_iter']
        })
    avg_c = np.mean([r['conflicts'] for r in results_all[mode]])
    avg_o = np.mean([r['obj']       for r in results_all[mode]])
    avg_t = np.mean([r['time']      for r in results_all[mode]])
    print(f"  {mode:8s} → conflits_moy={avg_c:.2f} "
          f"| obj_moy={avg_o:.2f} | temps_moy={avg_t:.3f}s")

print("\n✅ Comparaison terminée")

---
## 8. Expérimentations & Analyse des résultats

In [ ]:
# ============================================================
# 8.1 Tableau récapitulatif des performances
# ============================================================

rows = []
for mode in modes:
    data = results_all[mode]
    rows.append({
        'Mode'             : mode,
        'Conflits moy.'    : np.mean([r['conflicts'] for r in data]),
        'Conflits std.'    : np.std([r['conflicts']  for r in data]),
        'Conflits min.'    : np.min([r['conflicts']  for r in data]),
        'Objectif moy.'    : np.mean([r['obj']       for r in data]),
        'Temps moy. (s)'   : np.mean([r['time']      for r in data]),
        'Nb couleurs moy.' : np.mean([r['n_colors']  for r in data]),
    })

df_results = pd.DataFrame(rows).set_index('Mode')
print("Résultats moyens sur G_medium (10 runs) :")
display(df_results.round(3))

In [ ]:
# ============================================================
# 8.2 Courbes de convergence (1 run représentatif)
# ============================================================

random.seed(SEED); np.random.seed(SEED)

conv_results = {}
for mode in ['random', 'ml', 'kempe']:
    conv_results[mode] = tabu_search(
        G_medium, n_colors=N_COLORS, max_iter=MAX_ITER,
        tabu_tenure=10, operator_mode=mode,
        ml_model=rf_model if mode=='ml' else None,
        scaler=scaler   if mode=='ml' else None,
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_mode = {'random': 'gray', 'ml': 'crimson', 'kempe': 'steelblue'}

for mode in ['random', 'ml', 'kempe']:
    hist = conv_results[mode]['history']
    axes[0].plot(hist['obj'],       label=mode, color=colors_mode[mode], alpha=0.8)
    axes[1].plot(hist['conflicts'], label=mode, color=colors_mode[mode], alpha=0.8)

axes[0].set_title("Convergence — Objectif",  fontsize=12)
axes[0].set_xlabel("Itération")
axes[0].set_ylabel("Valeur objectif")
axes[0].legend()

axes[1].set_title("Convergence — Conflits", fontsize=12)
axes[1].set_xlabel("Itération")
axes[1].set_ylabel("Nombre de conflits")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8.3 Distribution des conflits — Boxplots
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot conflits
data_conflicts = [[r['conflicts'] for r in results_all[m]] for m in modes]
bp1 = axes[0].boxplot(data_conflicts, labels=modes, patch_artist=True)
palette = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B3']
for patch, color in zip(bp1['boxes'], palette):
    patch.set_facecolor(color)
axes[0].set_title("Distribution des conflits (10 runs)", fontsize=12)
axes[0].set_ylabel("Nombre de conflits")
axes[0].axhline(0, color='green', linestyle='--', alpha=0.5, label='Optimal')
axes[0].legend()

# Boxplot temps de calcul
data_times = [[r['time'] for r in results_all[m]] for m in modes]
bp2 = axes[1].boxplot(data_times, labels=modes, patch_artist=True)
for patch, color in zip(bp2['boxes'], palette):
    patch.set_facecolor(color)
axes[1].set_title("Temps de calcul (10 runs)", fontsize=12)
axes[1].set_ylabel("Temps (s)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8.4 Analyse de l'usage des opérateurs par le ML
# ============================================================

ml_ops = conv_results['ml']['history']['operators']
op_counts = {'recolor': ml_ops.count('recolor'),
             'swap'   : ml_ops.count('swap'),
             'kempe'  : ml_ops.count('kempe')}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Camembert
ax1.pie(op_counts.values(), labels=op_counts.keys(),
        autopct='%1.1f%%', colors=['steelblue','darkorange','green'],
        startangle=90)
ax1.set_title("Répartition des opérateurs choisis par ML", fontsize=11)

# Évolution temporelle
window = 30
op_series = pd.Series([{'recolor':0,'swap':1,'kempe':2}[o] for o in ml_ops])
for op_id, op_name, col in [(0,'recolor','steelblue'),
                              (1,'swap','darkorange'),
                              (2,'kempe','green')]:
    usage = (op_series == op_id).rolling(window).mean()
    ax2.plot(usage, label=op_name, color=col, alpha=0.8)
ax2.set_title(f"Utilisation des opérateurs (fenêtre={window} iter)",
              fontsize=11)
ax2.set_xlabel("Itération")
ax2.set_ylabel("Fréquence d'utilisation")
ax2.legend()

plt.tight_layout()
plt.show()

print("Répartition des opérateurs :")
total = sum(op_counts.values())
for op, cnt in op_counts.items():
    print(f"  {op:8s} : {cnt:4d} ({100*cnt/total:.1f}%)")

In [ ]:
# ============================================================
# 8.5 Comparaison sur plusieurs tailles de graphe
# ============================================================

test_instances = [
    ('G_small',  G_small,  15),
    ('G_medium', G_medium, 30),
    ('G_large',  G_large,  50),
    ('G_dsjc',   G_dsjc,   50),
]

comparison_rows = []
N_RUNS_COMP = 5

for name, G, _ in test_instances:
    for mode in ['random', 'ml']:
        conflicts_list, times_list = [], []
        for run in range(N_RUNS_COMP):
            random.seed(run); np.random.seed(run)
            res = tabu_search(
                G, n_colors=N_COLORS, max_iter=MAX_ITER,
                tabu_tenure=10, operator_mode=mode,
                ml_model=rf_model if mode=='ml' else None,
                scaler=scaler     if mode=='ml' else None,
            )
            conflicts_list.append(res['best_conflicts'])
            times_list.append(res['n_iter'])
        comparison_rows.append({
            'Instance' : name,
            'Mode'     : mode,
            'Conflits moy.' : np.mean(conflicts_list),
            'Conflits std.' : np.std(conflicts_list),
            'Iters moy.'    : np.mean(times_list),
        })

df_comp = pd.DataFrame(comparison_rows)
print("Comparaison Tabou Random vs Tabou+ML :")
display(df_comp.round(2))

In [ ]:
# ============================================================
# 8.6 Visualisation comparative par instance
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5))

instances = df_comp['Instance'].unique()
x         = np.arange(len(instances))
width     = 0.35

for i, mode in enumerate(['random', 'ml']):
    sub    = df_comp[df_comp['Mode'] == mode]
    values = [sub[sub['Instance']==inst]['Conflits moy.'].values[0]
              for inst in instances]
    errs   = [sub[sub['Instance']==inst]['Conflits std.'].values[0]
              for inst in instances]
    ax.bar(x + i*width, values, width, yerr=errs,
           label=f'Tabou {mode}',
           color='gray' if mode=='random' else 'crimson',
           alpha=0.8, capsize=4)

ax.set_xticks(x + width/2)
ax.set_xticklabels(instances)
ax.set_ylabel("Conflits moyens")
ax.set_title("Tabou Random vs Tabou+ML — Comparaison sur 4 instances",
             fontsize=12)
ax.legend()
ax.axhline(0, color='green', linestyle='--', alpha=0.4, label='Optimal')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8.7 Visualisation finale : meilleure solution ML
# ============================================================

random.seed(SEED); np.random.seed(SEED)
best_ml = tabu_search(
    G_medium, n_colors=N_COLORS, max_iter=MAX_ITER,
    tabu_tenure=10, operator_mode='ml',
    ml_model=rf_model, scaler=scaler
)

visualize_coloring(G_medium, best_ml['best_coloring'],
                   f"Meilleure solution — Tabou+ML (G_medium)")
print("Meilleure solution Tabou+ML :")
solution_info(G_medium, best_ml['best_coloring'])

---
## 9. Analyse, limites et pistes d'amélioration

### 9.1 Apports du Machine Learning

| Aspect | Observation |
|--------|-------------|
| **Qualité des solutions** | Le mode ML réduit en moyenne le nombre de conflits par rapport au mode aléatoire |
| **Stabilité** | La variance inter-runs est plus faible avec ML (moins de mauvaises surprises) |
| **Stratégie apprise** | Le ML apprend à préférer Kempe en phase de plateau et Recolor en phase de descente |
| **Temps de calcul** | Légèrement supérieur (prédiction ML), mais compensé par moins d'itérations nécessaires |

### 9.2 Limites

- **Généralisation** : le modèle est entraîné sur des graphes de 15–40 nœuds ; ses performances peuvent se dégrader sur des instances très différentes
- **Label bruité** : le label « meilleur opérateur » est calculé sur un seul pas en avant (greedy), pas sur le long terme
- **Features limitées** : on n'exploite pas la structure locale du graphe (degré des nœuds en conflit, voisinage, etc.)
- **Random Forest** : modèle statique, il ne s'adapte pas en ligne pendant la recherche

### 9.3 Pistes d'amélioration

1. **Apprentissage par renforcement (Q-Learning)** : apprendre dynamiquement quelle action prendre selon l'état, avec récompense = réduction des conflits
2. **Features de structure locale** : inclure le degré moyen des nœuds en conflit, la taille des composantes connexes par couleur
3. **Labeling multi-pas** : étiqueter avec l'opérateur qui mène au meilleur résultat après k itérations
4. **GNN (Graph Neural Networks)** : encoder directement la structure du graphe comme feature
5. **Ajustement dynamique du tabu tenure** : laisser le ML ajuster aussi ce paramètre

In [ ]:
# ============================================================
# 9.4 Résumé final
# ============================================================

print("=" * 65)
print("RÉSUMÉ FINAL DU PROJET")
print("=" * 65)
print()
print("Problème   : Coloration de graphe (cartes géographiques)")
print("Métaheur.  : Recherche Tabou")
print("ML         : Random Forest pour sélection d'opérateur")
print("Opérateurs : Recolor | Swap | Kempe Chain")
print()

for mode in modes:
    data = results_all[mode]
    avg_c = np.mean([r['conflicts'] for r in data])
    avg_t = np.mean([r['time']      for r in data])
    print(f"  Mode {mode:8s} → conflits_moy={avg_c:.2f} | temps_moy={avg_t:.3f}s")

print()
random_avg = np.mean([r['conflicts'] for r in results_all['random']])
ml_avg     = np.mean([r['conflicts'] for r in results_all['ml']])
if random_avg > 0:
    gain = (random_avg - ml_avg) / random_avg * 100
    print(f"→ Gain ML vs Random : {gain:.1f}% de conflits en moins")
print()
print("Précision du classifieur ML : {:.1f}%".format(score_test*100))